In [171]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
import os 
import joblib


In [140]:
df = pd.read_csv(r'D:\End-to-end-ML\Customer-Risk-Escalation-Engine\data\raw_data\support_tkts.csv')
df

,ticket_id,customer_name,customer_email,product,category,issue_description,resolution_notes,priority,status,channel,...,ticket_resolved_date,escalated,sla_breached,operating_system,browser,payment_method,language,preferred_contact_time,issue_complexity_score,customer_segment
0,1,Patricia Smith,patricia.smith760@outlook.com,Web Portal,Account Suspension,The payment was deducted from my bank account ...,Data synchronization restored after backend se...,Urgent,Open,Email,...,2023-05-20,No,Yes,MacOS,Edge,PayPal,French,Afternoon,4,Small Business
1,2,Patricia Williams,patricia.williams390@gmail.com,Mobile App,Performance Issue,I found a bug in the latest update affecting r...,Provided step-by-step troubleshooting instruct...,Urgent,Closed,Email,...,2024-01-19,Yes,Yes,Windows,Firefox,PayPal,English,Afternoon,2,Small Business
2,3,William Anderson,william.anderson651@outlook.com,Web Portal,Performance Issue,The application crashes whenever I try to uplo...,Provided step-by-step troubleshooting instruct...,Medium,Closed,Chat,...,2022-12-05,Yes,Yes,Windows,Safari,Bank Transfer,French,Morning,4,Corporate
3,4,David Miller,david.miller672@icloud.com,Payment Gateway,Subscription Cancellation,My subscription was cancelled without my reque...,Provided step-by-step troubleshooting instruct...,Medium,Closed,Social Media,...,2024-04-04,Yes,No,Windows,Chrome,Credit Card,Spanish,Afternoon,7,Corporate
4,5,Robert Gonzalez,robert.gonzalez391@hotmail.com,Web Portal,Feature Request,The system is not syncing data across devices ...,We have reset the account credentials and advi...,High,Pending Customer,Email,...,2024-08-24,Yes,No,Linux,NaN,Debit Card,Spanish,Evening,3,Corporate
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
199995,199996,Barbara Martinez,barbara.martinez691@hotmail.com,Billing System,Bug Report,There seems to be a discrepancy in my billing ...,Payment gateway timeout issue fixed and monito...,Urgent,Closed,Social Media,...,2024-01-16,No,No,Linux,Safari,Credit Card,French,Morning,7,Small Business
199996,199997,James Hernandez,james.hernandez188@hotmail.com,E-commerce Store,Login Issue,Two-factor authentication codes are not being ...,Provided step-by-step troubleshooting instruct...,Low,Open,Web Form,...,2022-05-18,No,Yes,MacOS,Safari,Bank Transfer,English,Evening,3,Individual
199997,199998,Michael Thomas,michael.thomas66@company.com,Web Portal,Refund Request,Two-factor authentication codes are not being ...,Provided step-by-step troubleshooting instruct...,Low,Closed,Chat,...,2022-04-30,Yes,No,Windows,Chrome,Bank Transfer,French,Morning,5,Small Business
199998,199999,Barbara Taylor,barbara.taylor486@outlook.com,Billing System,Performance Issue,There seems to be a discrepancy in my billing ...,Explained billing breakdown and clarified appl...,Urgent,Pending Customer,Social Media,...,2024-01-27,No,Yes,iOS,Chrome,PayPal,Spanish,Night,6,Small Business


In [141]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200000 entries, 0 to 199999
Data columns (total 30 columns):
 #   Column                       Non-Null Count   Dtype  
---  ------                       --------------   -----  
 0   ticket_id                    200000 non-null  int64  
 1   customer_name                200000 non-null  object 
 2   customer_email               200000 non-null  object 
 3   product                      200000 non-null  object 
 4   category                     200000 non-null  object 
 5   issue_description            200000 non-null  object 
 6   resolution_notes             200000 non-null  object 
 7   priority                     200000 non-null  object 
 8   status                       200000 non-null  object 
 9   channel                      200000 non-null  object 
 10  region                       200000 non-null  object 
 11  customer_age                 200000 non-null  int64  
 12  customer_gender              200000 non-null  object 
 13 

In [142]:
col = ['ticket_id','customer_name','customer_email','channel','region','customer_age','customer_gender','browser','language','preferred_contact_time','operating_system']

In [143]:
df = df.drop(col,axis=1)

In [144]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200000 entries, 0 to 199999
Data columns (total 19 columns):
 #   Column                       Non-Null Count   Dtype  
---  ------                       --------------   -----  
 0   product                      200000 non-null  object 
 1   category                     200000 non-null  object 
 2   issue_description            200000 non-null  object 
 3   resolution_notes             200000 non-null  object 
 4   priority                     200000 non-null  object 
 5   status                       200000 non-null  object 
 6   subscription_type            200000 non-null  object 
 7   customer_tenure_months       200000 non-null  int64  
 8   previous_tickets             200000 non-null  int64  
 9   customer_satisfaction_score  200000 non-null  int64  
 10  first_response_time_hours    200000 non-null  float64
 11  resolution_time_hours        200000 non-null  float64
 12  ticket_created_date          200000 non-null  object 
 13 

In [145]:
df['ticket_created_date'] = pd.to_datetime(
    df['ticket_created_date']
).dt.normalize()

df['ticket_resolved_date'] = pd.to_datetime(
    df['ticket_resolved_date']
).dt.normalize()

In [146]:
df['ticket_created_date'] .dtype

dtype('<M8[ns]')

In [147]:
df['ticket_created_date']

0        2023-05-17
1        2024-01-06
2        2022-11-30
3        2024-03-21
4        2024-08-16
            ...    
199995   2024-01-02
199996   2022-05-16
199997   2022-04-16
199998   2024-01-27
199999   2022-02-23
Name: ticket_created_date, Length: 200000, dtype: datetime64[ns]

In [148]:
df['created_year'] = df['ticket_created_date'].dt.year
df['created_month'] = df['ticket_created_date'].dt.month_name()
df['weekday_num'] = df['ticket_created_date'].dt.dayofweek

In [149]:
df['resolved_year'] = df['ticket_resolved_date'].dt.year
df['resolved_month'] = df['ticket_resolved_date'].dt.month
df['weekday_num'] = df['ticket_resolved_date'].dt.dayofweek


In [150]:
df['ticket_resolution_days'] = (df['ticket_resolved_date'] - df['ticket_created_date']).dt.days

In [151]:
df['ticket_resolution_days']

0          3
1         13
2          5
3         14
4          8
          ..
199995    14
199996     2
199997    14
199998     0
199999     5
Name: ticket_resolution_days, Length: 200000, dtype: int64

In [152]:
df['customer_satisfaction_score'].value_counts().sort_values()

customer_satisfaction_score
1    39756
5    39905
3    40082
4    40086
2    40171
Name: count, dtype: int64

In [153]:
df['is_dissatisfied'] = [1 if x<=1 else 0 for x in df['customer_satisfaction_score']]

In [154]:
df['issue_char_length']  = df['issue_description'].str.len()
df['issue_word_count']   = df['issue_description'].str.split().str.len()

In [155]:
df['resolution_char_length'] = df['resolution_notes'].str.len()
df['resolution_word_count']  = df['resolution_notes'].str.split().str.len()

In [156]:
df['issue_resolution_length_ratio'] = (
    df['issue_word_count'] / df['resolution_word_count']
).round(2)

In [157]:
df['combined_text'] = (
    df['issue_description'] + " " + df['resolution_notes']
)

In [158]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200000 entries, 0 to 199999
Data columns (total 32 columns):
 #   Column                         Non-Null Count   Dtype         
---  ------                         --------------   -----         
 0   product                        200000 non-null  object        
 1   category                       200000 non-null  object        
 2   issue_description              200000 non-null  object        
 3   resolution_notes               200000 non-null  object        
 4   priority                       200000 non-null  object        
 5   status                         200000 non-null  object        
 6   subscription_type              200000 non-null  object        
 7   customer_tenure_months         200000 non-null  int64         
 8   previous_tickets               200000 non-null  int64         
 9   customer_satisfaction_score    200000 non-null  int64         
 10  first_response_time_hours      200000 non-null  float64       
 11  

In [159]:
colum = ['issue_description','resolution_notes','ticket_created_date','ticket_resolved_date']
df = df.drop(colum,axis=1)


In [161]:
df['escalated']   = df['escalated'].map({'Yes': 1, 'No': 0})
df['sla_breached'] = df['sla_breached'].map({'Yes': 1, 'No': 0})
df['status']      = df['status'].map({'Resolved': 1, 'Open': 0})

In [162]:
priority_map = {
    'Low': 1, 'Medium': 2, 'High': 3, 'Urgent': 4
}
df['priority'] = df['priority'].map(priority_map)

In [163]:
le = LabelEncoder()
cat_cols = [
    'product', 'category', 'subscription_type',
    'payment_method', 'customer_segment'
]
for col in cat_cols:
    df[col] = le.fit_transform(df[col])

print("Encoding done ✅")
print(df.dtypes)

Encoding done ✅
product                            int32
category                           int32
priority                           int64
status                           float64
subscription_type                  int32
customer_tenure_months             int64
previous_tickets                   int64
customer_satisfaction_score        int64
first_response_time_hours        float64
resolution_time_hours            float64
escalated                          int64
sla_breached                       int64
payment_method                     int32
issue_complexity_score             int64
customer_segment                   int32
created_year                       int32
created_month                     object
weekday_num                        int32
resolved_year                      int32
resolved_month                     int32
ticket_resolution_days             int64
is_dissatisfied                    int64
issue_char_length                  int64
issue_word_count                   int64


In [164]:
df['escalated']

0         0
1         1
2         1
3         1
4         1
         ..
199995    0
199996    0
199997    1
199998    0
199999    0
Name: escalated, Length: 200000, dtype: int64

## Train_Test_Split

In [165]:
combined_text = df['combined_text'].copy()

In [166]:
y = df['escalated']
X = df.drop(columns=[
    'escalated',
    'combined_text'
])

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (200000, 26)
y shape: (200000,)


In [168]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

# Split combined_text the same way
text_train, text_test = train_test_split(
    combined_text,
    test_size=0.2,
    random_state=42
)

print(f"X_train : {X_train.shape}")
print(f"X_test  : {X_test.shape}")
print(f"y_train : {y_train.shape}")
print(f"y_test  : {y_test.shape}")

X_train : (150000, 26)
X_test  : (50000, 26)
y_train : (150000,)
y_test  : (50000,)


In [169]:
scale_cols = [
    'customer_tenure_months',
    'previous_tickets',
    'customer_satisfaction_score',
    'first_response_time_hours',
    'resolution_time_hours',
    'issue_complexity_score',
    'ticket_resolution_days',
    'issue_char_length',
    'issue_word_count',
    'resolution_char_length',
    'resolution_word_count',
    'issue_resolution_length_ratio'
]

scaler = StandardScaler()
X_train[scale_cols] = scaler.fit_transform(X_train[scale_cols])

X_test[scale_cols] = scaler.transform(X_test[scale_cols])

In [174]:
os.makedirs('data/processed', exist_ok=True)
os.makedirs('models/saved_models', exist_ok=True)

X_train.to_csv('data/processed/X_train.csv', index=False)
X_test.to_csv('data/processed/X_test.csv',   index=False)
y_train.to_csv('data/processed/y_train.csv', index=False)
y_test.to_csv('data/processed/y_test.csv',   index=False)

text_train.to_csv('data/processed/text_train.csv', index=False)
text_test.to_csv('data/processed/text_test.csv',   index=False)

joblib.dump(scaler, 'models/saved_models/scaler.pkl')

print("All files saved ✅")

All files saved ✅
